routers/commentaires.py

In [ ]:
import re
import os
from enum import Enum
from typing import Optional, List, Dict
from fastapi import APIRouter, Query, Depends, HTTPException
from routers.auth import guard
from services.mongo_service import get_last_comments, list_societes, db
from bson import ObjectId
from re import findall, escape


-----------------------
Router principal
-----------------------

In [ ]:
router = APIRouter(prefix="/commentaires", tags=["Commentaires"])


-----------------------
Enum des sociétés connues (pour Swagger)
-----------------------

In [ ]:
class SocieteEnum(str, Enum):
    temu = "temu.com"
    tesla = "tesla.com"
    chronopost = "chronopost.fr"
    vinted = "vinted.fr"


-----------------------
Derniers commentaires
-----------------------

Retourne les derniers commentaires enregistrés dans MongoDB.

In [ ]:
@router.get("/last")
def fetch_last_comments(
    societe_id: SocieteEnum | None = Query(None, description="Choisir une société"),
    limit: int = Query(100, ge=1, le=1000, description="Nombre max de commentaires à retourner"),
    skip: int = Query(0, ge=0, description="Décalage pour la pagination"),
    user=Depends(guard("full_only")),  # 🔒 activé uniquement si AUTH_MODE=full
):
    return {"comments": get_last_comments(societe_id=societe_id, limit=limit, skip=skip)}


-----------------------
Liste des sociétés
-----------------------

Retourne la liste des sociétés avec leur note globale.

In [ ]:
@router.get("/societes")
def fetch_societes(
    limit: int = Query(1000, ge=1, le=5000),
    user=Depends(guard("full_only")),
):
    return {"societes": list_societes(limit=limit)}


-----------------------
Top Avis
-----------------------

Retourne les avis les plus positifs/négatifs.

In [ ]:
@router.get("/top_avis")
def fetch_top_avis(
    societe_id: str | None = Query(None, description="Filtre société: accepte 'temu' ou 'temu.com', etc."),
    limit: int = Query(10, ge=1, le=100, description="Nombre d'avis à retourner"),
    positif: bool = Query(True, description="True = meilleurs avis (note=5), False = pires avis (note=1)"),
    user=Depends(guard("full_only")),
):
    collection = db["avis_trustpilot"]


    # ----- Filtre société -----

In [ ]:
    base_query: Dict = {}
    if societe_id:
        pat = re.escape(societe_id.strip())
        base_query["$or"] = [
            {"id_societe": {"$regex": pat, "$options": "i"}},
            {"societe_nom": {"$regex": pat, "$options": "i"}},
        ]


    # ----- Filtre de note -----

In [ ]:
    target = 5 if positif else 1
    note_or = [
        {"note": target},
        {"note_commentaire": target},
        {"note_commentaire": {"$regex": rf"^\s*{target}\b", "$options": "i"}},
    ]
    strict_query = {"$and": [base_query, {"$or": note_or}]} if base_query else {"$or": note_or}

    sort_order = -1 if positif else 1
    cursor = collection.find(strict_query).sort(
        [("note", sort_order), ("note_commentaire", sort_order)]
    ).limit(limit)

    docs = list(cursor)
    if not docs:
        cursor = collection.find(base_query or {}).sort(
            [("note", sort_order), ("note_commentaire", sort_order)]
        ).limit(limit)
        docs = list(cursor)

    avis = [
        {
            "id": str(d.get("_id")),
            "auteur": d.get("auteur"),
            "commentaire": d.get("commentaire"),
            "note": d.get("note_commentaire") or d.get("note"),
            "date": d.get("date"),
            "societe": d.get("societe_nom") or d.get("id_societe"),
        }
        for d in docs
    ]
    return {"top_avis": avis}


-----------------------
Recherche par mots/phrases (+ filtre société)
-----------------------

    
    
    
    
Endpoint factice pour tests unitaires.

In [ ]:
@router.get("/search")
def search_commentaires(
    q: str = Query(..., description='Un ou plusieurs mots/phrases. Mettez une phrase entre guillemets pour une recherche exacte, ex: "Plus jamais".'),
    societe: str | None = Query(None, description="Ex: temu, temu.com, vinted, vinted.fr"),
    limit: int = Query(50, ge=1, le=1000),
    match_all: bool = Query(False, description="True = tous les tokens doivent être présents (ET). False = au moins un (OU)."),
):
    return [
        {"id": 1, "texte": "Super service", "note": 5},
        {"id": 2, "texte": "Pas terrible", "note": 2},
    ]
